# Supervised Economic Valuation of Clusters (Market Margins V2)

This enhanced notebook anchors cluster propensity scores to real-world Italian retail banking margins. It calculates the tangible € monetary Customer Lifetime Value (CLV), applies multi-channel service costs, and provides sensitivity analysis.

**Sections:**
1. Market margin table & Direct Revenue Estimation
2. Revenue ranking chart
3. Propensity heatmap with revenue overlay
4. Sensitivity analysis (Tornado chart)
5. Channel cost adjustment
6. Board-ready export

## Setup & Data Load
Load the preprocessed data, map the weighted clusters, and quickly compute product propensities via Logistic Regression.

In [ ]:
import numpy as np
import pandas as pd
import pickle
import logging
from pathlib import Path
import plotly.express as px
import plotly.graph_objects as go
import seaborn as sns
import matplotlib.pyplot as plt
from sklearn.preprocessing import LabelEncoder
from sklearn.linear_model import LogisticRegression
from sklearn.model_selection import train_test_split
from openpyxl import load_workbook
from openpyxl.formatting.rule import ColorScaleRule

logging.basicConfig(level=logging.INFO, format="%(asctime)s - %(message)s")
logger = logging.getLogger(__name__)

# Load Data
df_raw = pd.read_excel(r"./Data/Dataset1_BankClients.xlsx")
df = df_raw.drop(columns=["ID"])

# Handle Missing Values
categorical_cols = ["Gender", "Job", "Area"]
numerical_cols = [c for c in df.columns if c not in categorical_cols]

for col in numerical_cols:
    df[col] = df[col].fillna(df[col].median())
for col in categorical_cols:
    df[col] = df[col].fillna(df[col].mode()[0])

# Minor filter
mask_minors = (df["Age"] < 18) & (df["Job"].isin([2, 3, 4]))
df = df[~mask_minors].reset_index(drop=True)

# Load Cluster Labels
RESULTS_DIR = Path("results")
with open(RESULTS_DIR / "weighted_results.pkl", "rb") as f:
    weighted_res = pickle.load(f)
    
df["Cluster"] = weighted_res["winner_lbls"]
logger.info(f"Loaded {len(df)} records with cluster labels.")

# Pre-compute Product Propensities using Logistic Regression
features = [c for c in df.columns if c != "Cluster"]
df_dummies = pd.get_dummies(df[features], columns=categorical_cols, drop_first=True)
X = df_dummies.values

product_targets = {
    "Savings": (df["Saving"] > df["Saving"].median()).astype(int),
    "Investments": (df["Investments"] > df["Investments"].median()).astype(int),
    "ESG": (df["ESG"] > df["ESG"].median()).astype(int),
    "Mortgage": ((df["Debt"] > df["Debt"].median()) & (df["FamilySize"] > df["FamilySize"].median())).astype(int),
    "Digital": (df["Digital"] > df["Digital"].median()).astype(int)
}

df_prop = pd.DataFrame({'Cluster': df['Cluster']})
for target_name, y_binary in product_targets.items():
    clf = LogisticRegression(max_iter=1000, random_state=42)
    clf.fit(X, y_binary)
    df_prop[f"{target_name}_Prop"] = clf.predict_proba(X)[:, 1]



## Section 1: Market Margin Table
Compute Expected Annual Revenue mapping propensities against scaled AUM/Loan volumes and realistic margin bounds.

In [ ]:
# Section 1 - Market Margin Table & Direct Revenue Estimation
# Assumptions based on Italian retail banking environment
product_margins = {
    "Savings": 0.018,
    "Investments": 0.012,
    "ESG": 0.014,
    "Mortgage": 0.015
}

# AUM / Loan average proxies per product (in Euros)
aum_proxies = {
    "Savings": 50000,
    "Investments": 60000,
    "ESG": 60000,
    "Mortgage": 150000
}

def compute_revenue(row, margins, proxies):
    rev = 0
    for prod in margins.keys():
        rev += row[f"{prod}_Prop"] * proxies[prod] * margins[prod]
    return rev

df_prop['Expected_Annual_Revenue'] = df_prop.apply(compute_revenue, args=(product_margins, aum_proxies), axis=1)

# Aggregate to Cluster Level
cluster_revenue = df_prop.groupby('Cluster').agg(
    Size=('Cluster', 'count'),
    Per_Capita_Revenue=('Expected_Annual_Revenue', 'mean'),
    Total_Revenue=('Expected_Annual_Revenue', 'sum')
).reset_index()

cluster_revenue = cluster_revenue.sort_values(by='Total_Revenue', ascending=False).reset_index(drop=True)
display(cluster_revenue.round(0))

## Section 2: Revenue Ranking Chart
Visualize Total Addressable Revenue (TAR) by strategic group and export formatted ranking.

In [ ]:
# Section 2 - Revenue Ranking Chart
# Define Strategic Groups
strats = {
    5: "Volume Engines", 6: "Volume Engines",
    0: "Stable Core", 2: "Stable Core",
    8: "Family Borrowers", 9: "Family Borrowers",
    3: "ESG Retirees", 7: "ESG Retirees",
    1: "Transitional", 4: "Transitional"
}
cluster_revenue['Strategic_Group'] = cluster_revenue['Cluster'].map(strats)

fig = px.bar(
    cluster_revenue, x='Total_Revenue', y='Cluster', orientation='h',
    color='Strategic_Group', 
    text=cluster_revenue['Per_Capita_Revenue'].apply(lambda x: f"€{x:,.0f} per capita"),
    title="Cluster Revenue Ranking (Total Addressable Value)",
    labels={'Total_Revenue': 'Total Annual Revenue (€)', 'Cluster': 'Cluster ID'}
)
fig.update_layout(yaxis=dict(type='category', autorange="reversed"))
fig.show()

# Export with Openpyxl formatting
output_path = RESULTS_DIR / "cluster_revenue_ranking_v2.xlsx"
with pd.ExcelWriter(output_path, engine='openpyxl') as writer:
    cluster_revenue.to_excel(writer, index=False, sheet_name="Ranking")
    worksheet = writer.sheets['Ranking']
    
    # Simple conditional formatting rule via openpyxl
    rule = ColorScaleRule(start_type='min', start_color='F8696B',
                          mid_type='percentile', mid_value=50, mid_color='FFEB84',
                          end_type='max', end_color='63BE7B')
    worksheet.conditional_formatting.add(f'D2:D{len(cluster_revenue)+1}', rule)

logger.info(f"Saved ranked revenue to {output_path}")

## Section 3: Propensity Heatmap with Revenue Overlay
A heatmap coloured by propensity but annotated with tangible expected annual monetary revenue.

In [ ]:
# Section 3 - Propensity Heatmap with Revenue Overlay
prop_cols = ['Savings_Prop', 'Investments_Prop', 'ESG_Prop', 'Mortgage_Prop']
heatmap_data = df_prop.groupby('Cluster')[prop_cols].mean()

# Compute the equivalent revenue overlay values
revenue_overlay = pd.DataFrame(index=heatmap_data.index)
for prod in prop_cols:
    base_prod = prod.split('_')[0]
    revenue_overlay[prod] = heatmap_data[prod] * aum_proxies[base_prod] * product_margins[base_prod]

plt.figure(figsize=(10, 6))
sns.heatmap(heatmap_data, annot=revenue_overlay, fmt=".0f", cmap="RdYlGn", 
            cbar_kws={'label': 'Propensity Score [0-1]'})
plt.title("Cluster x Product Heatmap (Annotated with Expected € Revenue)")
plt.ylabel("Cluster")
plt.xlabel("Product Family")
plt.show()

# Export
heatmap_data.join(revenue_overlay, rsuffix='_ExpectedEUR').to_excel(RESULTS_DIR / "product_propensity_heatmap_v2.xlsx")

## Section 4: Sensitivity Analysis
Sweeping each margin rate $\pm 30$ basis points to understand macroeconomic exposure (e.g., ECB rate compression).

In [ ]:
# Section 4 - Sensitivity Analysis (Tornado Chart)
base_total_revenue = cluster_revenue['Total_Revenue'].sum()
sensitivity_results = []

sweep_bps = 0.0030 # +/- 30 basis points

for prod in product_margins.keys():
    # Downside
    down_margin = product_margins.copy()
    down_margin[prod] -= sweep_bps
    down_rev = df_prop.apply(compute_revenue, args=(down_margin, aum_proxies), axis=1).sum()
    
    # Upside
    up_margin = product_margins.copy()
    up_margin[prod] += sweep_bps
    up_rev = df_prop.apply(compute_revenue, args=(up_margin, aum_proxies), axis=1).sum()
    
    sensitivity_results.append({
        'Product': prod,
        'Downside ( -30 bps )': down_rev - base_total_revenue,
        'Upside ( +30 bps )': up_rev - base_total_revenue
    })

df_sens = pd.DataFrame(sensitivity_results).set_index("Product")
df_sens = df_sens.sort_values(by="Downside ( -30 bps )")

# Tornado Chart
plt.figure(figsize=(8, 5))
plt.barh(df_sens.index, df_sens['Downside ( -30 bps )'], color='salmon', label='-30 bps Margin')
plt.barh(df_sens.index, df_sens['Upside ( +30 bps )'], color='lightgreen', label='+30 bps Margin')
plt.axvline(0, color='black', linewidth=1)
plt.title("Revenue Sensitivity to Margin Compression/Expansion (±30 bps)")
plt.xlabel("Expected Change in Total Portfolio Revenue (€)")
plt.legend()
plt.show()

## Section 5: Channel Cost Adjustment
Subtracting estimated service costs using multiplier adjustments (Digital vs Hybrid vs Branch) to compute actual Net Margin.

In [ ]:
# Section 5 - Channel Cost Adjustment
# Base service cost assumed at 300 EUR per client annually for branch
base_service_cost = 300 

def apply_service_cost(row):
    cluster = row['Cluster']
    if cluster in [5, 6]:
        cost = base_service_cost * 0.70 # Digital-native
    elif cluster in [3, 7]:
        cost = base_service_cost * 1.00 # Branch-dependent
    else:
        cost = base_service_cost * 0.85 # Hybrid
    
    return row['Per_Capita_Revenue'] - cost

cluster_revenue['Per_Capita_Net_Margin'] = cluster_revenue.apply(apply_service_cost, axis=1)
cluster_revenue['Total_Net_Margin'] = cluster_revenue['Per_Capita_Net_Margin'] * cluster_revenue['Size']
cluster_revenue = cluster_revenue.sort_values(by='Total_Net_Margin', ascending=False).reset_index(drop=True)

display(cluster_revenue[['Cluster', 'Strategic_Group', 'Per_Capita_Revenue', 'Per_Capita_Net_Margin', 'Total_Net_Margin']].round(0))

## Section 6: Board-Ready Export
Generating the consolidated 3-sheet Excel workbook for commercial strategy delivery.

In [ ]:
# Section 6 - Board-ready Export
export_path = RESULTS_DIR / "board_ready_export.xlsx"
with pd.ExcelWriter(export_path, engine='openpyxl') as writer:
    # Sheet 1: Net Margin Ranking
    cluster_revenue.to_excel(writer, sheet_name="Net Margin Ranking", index=False)
    
    # Sheet 2: Heatmap
    heatmap_combined = heatmap_data.join(revenue_overlay, rsuffix='_ExpectedEUR')
    heatmap_combined.to_excel(writer, sheet_name="Product Propensity Heatmap", index=True)
    
    # Sheet 3: Sensitivity
    df_sens.to_excel(writer, sheet_name="Sensitivity Bounds", index=True)
    
logger.info(f"Final interactive board deck exported to {export_path}")